# FAIR-Chem / UMA · molecular energy and force fine-tuning

Adapt UMA-S to a small rMD17 ethanol subset using the supported FAIR-Chem trainer. All data conversion, configuration, launch, evaluation and plotting code is visible below. This uses molecular structures and DFT energy/force targets, not SMILES.

Use a separate Python 3.12 environment with an NVIDIA GPU. First request access to [facebook/UMA](https://huggingface.co/facebook/UMA). Provide an approved read token through `HF_TOKEN`, Hugging Face login, or `HF_TOKEN_FILE`; never paste it into a notebook. No weights are redistributed.

rMD17 uses PBE/def2-SVP, whereas UMA's `omol` head has a different reference method. This recipe is a small transfer experiment across levels of theory, not a like-for-like benchmark. All splits describe the same molecule; pretraining overlap and trajectory correlations are not excluded.


Run the environment and output directory on a persistent disk with at least 30 GiB free. The notebook stores model/compiler caches and temporary files under its output directory. On Kubernetes, mount a persistent volume there and set ephemeral-storage requests/limits; GPU memory capacity does not prevent disk-pressure eviction.


In [ ]:
import os,sys,subprocess,json,hashlib,urllib.request,time,re,ast,platform,importlib.metadata
from pathlib import Path
work=Path(os.environ.get('RECIPE_OUTPUT_DIR','results/uma')).resolve();work.mkdir(parents=True,exist_ok=True)
# Keep caches and temporary files beside the persistent output directory.
for variable,relative in [('HF_HOME','.cache/huggingface'),('TORCH_HOME','.cache/torch'),('TRITON_CACHE_DIR','.cache/triton'),('TMPDIR','.tmp')]:
    path=work/relative;path.mkdir(parents=True,exist_ok=True)
    os.environ[variable]=str(path)
os.environ['PIP_NO_CACHE_DIR']='1'
import tempfile,shutil
tempfile.tempdir=None
assert shutil.disk_usage(work).free >= 30 * 1024**3, "Use an output disk with at least 30 GiB free for dependencies, weights and checkpoints."
SOURCE_REV='645af7d03e91138ef07e51de387316c9e2e152dc'
import shutil
assert shutil.which('git') and shutil.which('cc'), 'Install Git and a C compiler (Debian/Ubuntu: apt-get install git build-essential).'
source=work/'fairchem-source'
source.mkdir(exist_ok=True)
if not (source/'.git').exists():
    subprocess.check_call(['git','init',str(source)])
    subprocess.check_call(['git','-C',str(source),'remote','add','origin','https://github.com/facebookresearch/fairchem.git'])
if subprocess.run(['git','-C',str(source),'rev-parse','--verify','HEAD'],capture_output=True).returncode != 0:
    subprocess.check_call(['git','-C',str(source),'fetch','--depth','1','origin',SOURCE_REV])
    subprocess.check_call(['git','-C',str(source),'checkout','--detach','FETCH_HEAD'])
assert subprocess.check_output(['git','-C',str(source),'rev-parse','HEAD'],text=True).strip()==SOURCE_REV
if os.environ.get('RECIPE_SKIP_INSTALL')!='1':
    subprocess.check_call([sys.executable,'-m','pip','install','-e',str(source/'packages/fairchem-core'),'matplotlib'])
# Editable-install .pth files are only loaded when Python starts; make this
# pinned checkout importable in the current notebook kernel too.
sys.path.insert(0,str(source/'src'))
import importlib
importlib.invalidate_caches()
import torch,numpy as np,yaml
from ase import Atoms
from ase.io import write
from ase.calculators.singlepoint import SinglePointCalculator
from ase.units import kcal,mol
from huggingface_hub import hf_hub_download,get_token
assert torch.cuda.is_available(),'Select a CUDA kernel.'
if os.environ.get('HF_TOKEN_FILE'):
    os.environ['HF_TOKEN']=Path(os.environ['HF_TOKEN_FILE']).read_text().strip()
assert get_token(),'Configure an approved HF read token outside the notebook.'
MODEL_REV='c5107e43d5e811dd7cb56f57533dc0eb550dee6b'
checkpoint=hf_hub_download('facebook/UMA','checkpoints/uma-s-1p1.pt',revision=MODEL_REV)


## Download pinned rMD17 splits and convert units
Use 32 training frames, eight validation frames from the training file, and eight frames from the separate test file. Original frame IDs must not overlap. Energies and forces are converted from kcal/mol to eV and eV/Å.


In [ ]:
DATA_REV='4f6bb0a186822c90d47cd98b7a27991fe0b3173b'
hashes={'train':'096946747b8d9dcfa042b44f8debbd3da387dfe1586b659e3be789cc4927316a',
        'test':'c3594a0f93d5992e02f9eb671392fb2e5021b3fa4f902417886966d1e4f64e1e'}
data={}
for split in ['train','test']:
    url=f'https://raw.githubusercontent.com/andersx/rmd17-npz/{DATA_REV}/rmd17-npz/rmd17_ethanol_{split}_01.npz'
    raw=urllib.request.urlopen(url,timeout=60).read();assert hashlib.sha256(raw).hexdigest()==hashes[split]
    path=work/f'{split}.npz';path.write_bytes(raw);data[split]=np.load(path,allow_pickle=False)
train_indices=list(range(32));val_indices=list(range(800,808));test_indices=list(range(8))
ids={'train':data['train']['old_indices'][train_indices].tolist(),
     'validation':data['train']['old_indices'][val_indices].tolist(),
     'test':data['test']['old_indices'][test_indices].tolist()}
for a,b in [('train','validation'),('train','test'),('validation','test')]:assert set(ids[a]).isdisjoint(ids[b])
(work/'split.json').write_text(json.dumps(ids,indent=2))
def structures(split,indices):
    d=data[split];items=[]
    for i in indices:
        atoms=Atoms(numbers=d['nuclear_charges'],positions=d['coords'][i],pbc=False)
        atoms.info.update(charge=0,spin=1)
        atoms.calc=SinglePointCalculator(atoms,energy=float(d['energies'][i])*(kcal/mol),forces=d['forces'][i]*(kcal/mol))
        items.append(atoms)
    return items
train_atoms=structures('train',train_indices);val_atoms=structures('train',val_indices);test_atoms=structures('test',test_indices)
for name,items in [('train',train_atoms),('validation',val_atoms)]:
    directory=work/name;directory.mkdir(exist_ok=True)
    write(directory/'structures.extxyz',items,format='extxyz')


## Prepare a fresh training configuration
The pinned official helper converts ASE structures into training databases. The generated YAML is shown and saved. The base checkpoint is explicitly pinned; fine-tuning runs locally on one GPU.


In [ ]:
# Unique output directory makes rerunning the preparation cell safe.
import uuid
prepared=work/('prepared-'+uuid.uuid4().hex[:8])
subprocess.check_call([sys.executable,str(source/'src/fairchem/core/scripts/create_uma_finetune_dataset.py'),
    '--train-dir',str(work/'train'),'--val-dir',str(work/'validation'),
    '--output-dir',str(prepared),'--uma-task','omol','--regression-tasks','ef',
    '--base-model','uma-s-1p1','--num-workers','1'],cwd=source)
config_path=prepared/'uma_sm_finetune_template.yaml'
config=yaml.safe_load(config_path.read_text())
config.update(epochs=2,steps=None,batch_size=2,lr=1e-5,evaluate_every_n_steps=8)
config['job']['run_dir']=str(work/'runs');config['job']['run_name']='uma-ethanol'
config['job']['debug']=False
config['job']['logger']['project']=os.environ.get('WANDB_PROJECT','open-chem-recipes')
config['job']['logger']['entity']=os.environ.get('WANDB_ENTITY')
unit=config['runner']['train_eval_unit'];unit['model']['checkpoint_location']=checkpoint;unit['print_every']=1
config.pop('base_model_name',None)  # Explicit pinned checkpoint replaces this template variable.
config_path.write_text(yaml.safe_dump(config,sort_keys=False))
print(yaml.safe_dump(config,sort_keys=False))


## Fine-tune with the official trainer
W&B defaults to offline and checkpoints stay local. The loss trace is also extracted from the trainer output. A failed subprocess stops the notebook; it does not fabricate results.


In [ ]:
environment=os.environ.copy();environment.setdefault('WANDB_MODE','online' if os.environ.get('WANDB_API_KEY') else 'offline')
started=time.monotonic();loss_curve=[]
command=[str(Path(sys.executable).parent/'fairchem'),'-c',str(config_path)]
with (work/'training.log').open('w') as log:
    process=subprocess.Popen(command,cwd=source,env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    for line in process.stdout:
        log.write(line);log.flush()
        if "'train/loss'" in line:
            match=re.search(r"\{.*'train/loss'.*\}",line)
            if match:
                try:
                    values=ast.literal_eval(match.group(0))
                    loss_curve.append({'step':int(values['train/step']),'loss':float(values['train/loss'])})
                except (ValueError,SyntaxError,KeyError):pass
    returncode=process.wait()
assert returncode==0,f'Training failed (exit {returncode}); inspect {work / "training.log"}.'
assert loss_curve,'No loss points parsed; inspect trainer output before reporting a run.'
trained=sorted((work/'runs').rglob('inference_ckpt.pt'),key=lambda p:p.stat().st_mtime)[-1]


## Compare energies and forces on unseen frames
Apply a constant energy correction fitted only on training frames, separately for the base and adapted model. Report corrected energy MAE and unmodified force MAE; do not mix energy reference methods without this distinction.


In [ ]:
from fairchem.core.units.mlip_unit import load_predict_unit
from fairchem.core import FAIRChemCalculator
import gc

def evaluate(path):
    predictor=load_predict_unit(str(path),device='cuda')
    calculator=FAIRChemCalculator(predictor,task_name='omol')
    def predict(items):
        energies=[];forces=[]
        for original in items:
            atoms=original.copy();atoms.calc=calculator
            energies.append(atoms.get_potential_energy());forces.append(atoms.get_forces())
        return np.asarray(energies),np.asarray(forces)
    train_energy,_=predict(train_atoms)
    train_reference=np.asarray([a.get_potential_energy() for a in train_atoms])
    offset=float(np.mean(train_reference-train_energy))
    energy,forces=predict(test_atoms)
    true_energy=np.asarray([a.get_potential_energy() for a in test_atoms])
    true_forces=np.asarray([a.get_forces() for a in test_atoms])
    scores={'energy_mae_ev_after_train_offset':float(np.mean(np.abs(energy+offset-true_energy))),
            'force_mae_ev_per_angstrom':float(np.mean(np.abs(forces-true_forces))),
            'train_energy_offset_ev':offset}
    del calculator,predictor;gc.collect();torch.cuda.empty_cache()
    return scores
before=evaluate(checkpoint);after=evaluate(trained)
metrics={'before':before,'after':after,'curve':loss_curve,'gpu':torch.cuda.get_device_name(0),
         'wall_seconds':time.monotonic()-started,'split_sizes':{k:len(v) for k,v in ids.items()}}
result={'metrics':metrics,'source_revision':SOURCE_REV,'model_revision':MODEL_REV,'dataset_revision':DATA_REV,
        'python':platform.python_version(),'packages':{p:importlib.metadata.version(p) for p in ['torch','fairchem-core','ase']}}
(work/'result.json').write_text(json.dumps(result,indent=2))
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,3,figsize=(13,3.5));axes[0].plot([p['step'] for p in loss_curve],[p['loss'] for p in loss_curve]);axes[0].set(title='UMA training loss',xlabel='Step',ylabel='Loss')
for ax,key,title in [(axes[1],'energy_mae_ev_after_train_offset','Energy MAE (eV, corrected)'),(axes[2],'force_mae_ev_per_angstrom','Force MAE (eV/Å)')]:
    ax.bar(['Base','Adapted'],[before[key],after[key]],color=['#64748b','#14b8a6']);ax.set_title(title)
fig.tight_layout();fig.savefig(work/'result.png',dpi=140);plt.show();print(json.dumps(metrics,indent=2))
